# Single-Band TX + Acquisition - ADRV9026

Transmit **one** waveform on a chosen TX and capture it on a chosen ORx, in one
programmed session.

**How to use:** edit the *Parameters* cell, then *Run All*. The session stays open
between cells; the last cell forces TX safe and disconnects.

Pipeline: choose profile -> show signal-path rates/bits -> load waveform ->
connect + program -> transmit -> capture one aligned snapshot -> plot / save.

## 1. Parameters (edit me)

**Level ranges**
- `TX_ATTEN_DB` - TX attenuation, **0.0 to 41.95 dB** in 0.05 dB steps. `0` = no attenuation (max power / strongest), `41.95` = max attenuation (weakest, the safe-state value). Higher number = quieter TX.
- `ORX_GAIN_INDEX` - ORx gain-table index, **0 to 255**, higher = more gain. Set manually below (the programmed default is 195; ADI examples use ~240). Raise it toward 255 for a hotter capture, lower it if the capture rails.


In [ ]:
from adrvtrx import TxChannel, RxChannel

# --- Profile -----------------------------------------------------------------
# Pick a friendly key below, OR paste any ".profile" filename / absolute path
# into PROFILE. The filename is resolved under the TES profiles folder.
PROFILES = {
    "98_linksharing":  "ADRV9025Init_StdUseCase98_LinkSharing.profile",   # Tx 491.52 MSPS, Np=12
    "102_linksharing": "ADRV9025Init_StdUseCase102_LinkSharing.profile",  # Np=16
    "14_linksharing":  "ADRV9025Init_StdUseCase14_LinkSharing.profile",
    "50_linksharing":  "ADRV9025Init_StdUseCase50_LinkSharing.profile",
}
PROFILE = "98_linksharing"          # <-- choose here

# --- Signal path: which TX transmits, which ORx observes it -------------------
# Bench wiring on this setup: TX2 -> ORx2, TX3 -> ORx3.
TX_CHANNEL  = TxChannel.TX2
ORX_CHANNEL = RxChannel.ORX2

# --- Input waveform (tab-delimited "I<TAB>Q", one complex sample per row) -----
SIGNAL_PATH = "C:/Users/ohammi/OneDrive - aus.edu/DualBand/input_100/Signal1.txt"

# --- Levels / capture --------------------------------------------------------
TX_ATTEN_DB     = 10.0    # 0.0-41.95 dB; 0 = full power, higher = quieter
CAPTURE_MS      = 0.1     # snapshot length in ms
NORMALIZE       = True    # scale waveform to full scale for the profile's bit depth
ORX_GAIN_INDEX  = 195     # ORx gain-table index, 0-255 (higher = more gain)
SAVE_CAPTURE_TO = None    # e.g. "captures/single_orx2.txt"; None = do not save

## 2. Imports, load config, choose profile

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from adrvtrx.config import load_config
from adrvtrx.profile import read_profile
from adrvtrx.radio import Radio
from adrvtrx.experiment import verify_status
from adrvtrx.waveform import load_tab_iq, full_scale
from adrvtrx.transmit import transmit_bands
from adrvtrx.capture import capture, measure_delay
from adrvtrx.gain import clip_report
from adrvtrx._enums import is_orx

cfg = load_config()
cfg.profile_name = PROFILES.get(PROFILE, PROFILE)   # friendly key or raw filename/path
info = read_profile(cfg.profile_path)
print("Profile:", cfg.profile_path.name)

## 3. Signal-path summary - sample rates & bit depth

In [ ]:
# Sample rate + bit depth per signal path. Prepare your TX waveform at the TX
# rate/bit-depth; a capture comes back at its path's rate (ORx runs faster than
# the main Rx in link-sharing profiles). full scale = 2**(Np-1) - 1.
def _fs(bits):
    return (1 << (bits - 1)) - 1

print(f"{'path':<6}{'rate (MSPS)':>14}{'bits (Np)':>12}{'full scale':>12}")
print(f"{'TX':<6}{info.tx_rate_khz/1000:>14.3f}{info.tx_bits:>12}{_fs(info.tx_bits):>12}")
print(f"{'Rx':<6}{info.rx_rate_khz/1000:>14.3f}{info.rx_bits:>12}{_fs(info.rx_bits):>12}")
print(f"{'ORx':<6}{info.orx_rate_khz/1000:>14.3f}{info.rx_bits:>12}{_fs(info.rx_bits):>12}")

CAP_RATE_HZ = info.orx_rate_hz if is_orx(ORX_CHANNEL) else info.rx_rate_hz
print(f"\nThis run: transmit {TX_CHANNEL.name} @ {info.tx_rate_khz/1000:.3f} MSPS / "
      f"{info.tx_bits}-bit; capture {ORX_CHANNEL.name} @ {CAP_RATE_HZ/1e6:.3f} MSPS / "
      f"{info.rx_bits}-bit")

## 4. Load the input waveform

In [ ]:
wave = load_tab_iq(SIGNAL_PATH)
print(f"loaded {len(wave)} samples from {SIGNAL_PATH}")
print(f"duration at TX rate: {len(wave)/info.tx_rate_hz*1e3:.3f} ms | "
      f"peak |x| = {np.max(np.abs(wave)):.1f}")

# Quick look at the input spectrum (before TX).
n = len(wave)
W = np.fft.fftshift(np.fft.fft(wave * np.hanning(n)))
f = np.fft.fftshift(np.fft.fftfreq(n, 1.0 / info.tx_rate_hz)) / 1e6
plt.figure(figsize=(8, 3))
plt.plot(f, 20 * np.log10(np.abs(W) / np.max(np.abs(W)) + 1e-12))
plt.title("Input waveform spectrum"); plt.xlabel("MHz"); plt.ylabel("dB")
plt.grid(True); plt.tight_layout(); plt.show()

## 5. Connect, program, verify

In [ ]:
# Opens the session and LEAVES IT OPEN for the cells below. TX is forced safe on
# connect; programming applies the profile, LOs, masks and tx->orx mapping.
# Run the final "Safe-state & disconnect" cell when you are done.
radio = Radio(cfg)
radio.connect()
radio.force_safe()
radio.program()
for k, v in verify_status(radio).items():
    print(f"{k}: {v}")

## 6. Transmit + capture (one aligned snapshot)

In [ ]:
radio.set_tx_atten(TX_CHANNEL, TX_ATTEN_DB)
radio.set_rx_gain(ORX_CHANNEL, ORX_GAIN_INDEX)   # ORx gain set manually (not auto-leveled)
transmit_bands(radio, {TX_CHANNEL: wave}, info.tx_bits,
               continuous=True, do_normalize=NORMALIZE)

# capture() enables the requested ORx input and reads its physical ADC slot.
result = capture(radio, int(ORX_CHANNEL), CAPTURE_MS, bits=info.rx_bits)
# TX->ORx delay vs the transmitted reference (captures 2x internally; TX must be live).
d_samp, d_ns, corr = measure_delay(radio, ORX_CHANNEL, wave, bits=info.rx_bits, fs=info.orx_rate_hz)
print(f"TX->ORx delay: {d_samp:.3f} samples ({d_ns:.1f} ns) | match corr {corr:.3f} "
      f"(low corr -> raise ORX_GAIN_INDEX for more ORx SNR)")
radio.disable_tx()   # stop the looping playback after the snapshot
print("captured channels:", [c.name for c in result.channels])

## 7. Inspect the capture (time + spectrum)

In [ ]:
cap = result.channels[ORX_CHANNEL]
ch = ORX_CHANNEL
rate = info.orx_rate_hz if is_orx(ch) else info.rx_rate_hz
iq = cap.i.astype(float) + 1j * cap.q.astype(float)
rep = clip_report(cap.i, cap.q, info.rx_bits)
print(f"{ch.name}: {len(iq)} samples @ {rate/1e6:.3f} MSPS | "
      f"peak {rep.peak_dbfs:+.1f} dBFS | railed {rep.railed_samples}")

fsfull = full_scale(info.rx_bits)
n = len(iq)
t_us = np.arange(n) / rate * 1e6
X = np.fft.fftshift(np.fft.fft(iq * np.hanning(n)))
fmhz = np.fft.fftshift(np.fft.fftfreq(n, 1.0 / rate)) / 1e6

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(t_us, iq.real / fsfull, label="I")
ax[0].plot(t_us, iq.imag / fsfull, label="Q")
ax[0].set_title(f"{ch.name} - time"); ax[0].set_xlabel("us"); ax[0].set_ylabel("norm")
ax[0].legend(loc="upper right"); ax[0].grid(True)
ax[1].plot(fmhz, 20 * np.log10(np.abs(X) / np.max(np.abs(X)) + 1e-12))
ax[1].set_title(f"{ch.name} - spectrum"); ax[1].set_xlabel("MHz"); ax[1].set_ylabel("dB")
ax[1].grid(True)
plt.tight_layout(); plt.show()

## 8. (optional) Save the capture
Writes normalized float I\tQ (divided by full scale).

In [ ]:
if SAVE_CAPTURE_TO:
    from pathlib import Path
    Path(SAVE_CAPTURE_TO).parent.mkdir(parents=True, exist_ok=True)
    result.channels[ORX_CHANNEL].save(SAVE_CAPTURE_TO)
    print("saved", SAVE_CAPTURE_TO)
else:
    print("SAVE_CAPTURE_TO is None - not saving")

## 9. Safe-state & disconnect (run when finished)

In [ ]:
radio.safe_state()     # max attenuation + clear TX enable
radio.disconnect()
print("TX safe, board disconnected")